# Mistral-7B — Zero-Shot Multi-Event Extraction

Extracts structured events (cultural, sports, mobility) from German and English news articles using Mistral-7B Instruct (4-bit quantized).

**Usage**: Place this notebook in the same folder as the dataset files:
- `events_2k_dataset.jsonl` — plain articles
- `events_2k_dataset_annotated.jsonl` — gold annotations
- `events_2k_dataset_failures.jsonl` — excluded annotation failures

Then run all cells sequentially. Outputs are saved to `./outputs/`.


In [ ]:
# ====== Configuration ======
PROJECT_NAME = "mistral_7b_events_2k"

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
USE_4BIT   = True

BATCH_SIZE         = 64
MAX_NEW_TOKENS     = 1200
TEMPERATURE        = 0.0
TOP_P              = 1.0
REPETITION_PENALTY = 1.05
MAX_INPUT_CHARS    = None

N_SAMPLES = None
SEEDS     = [42, 1, 2]
SEED      = SEEDS[0]

MODEL_TAG = "mistral_7b_events_2k"


In [ ]:
# ====== Install dependencies ======
!pip -q install -U "transformers>=4.44,<5.3" accelerate bitsandbytes huggingface_hub tqdm pandas numpy scikit-learn


In [ ]:
# ====== Imports ======
import os, json, re, random, datetime
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm


In [ ]:
# ====== Folder structure ======
from pathlib import Path

BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "data"
OUT_DIR  = BASE_DIR / "outputs"
PRED_DIR = OUT_DIR / "predictions"
MET_DIR  = OUT_DIR / "metrics"

for d in [DATA_DIR, OUT_DIR, PRED_DIR, MET_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR.resolve())


In [ ]:
# ====== Hugging Face authentication ======
# Set HF_TOKEN as an environment variable before running:
#   export HF_TOKEN="hf_..."   (Linux/Mac)
#   $env:HF_TOKEN = "hf_..."   (PowerShell)

HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    raise EnvironmentError(
        "HF_TOKEN not found. Set it as an environment variable before running."
    )

from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)
print("HF login OK")


In [ ]:
# ====== Data paths ======
PLAIN_PATH    = DATA_DIR / "events_2k_dataset.jsonl"
GOLD_PATH     = DATA_DIR / "events_2k_dataset_annotated.jsonl"
FAILURES_PATH = DATA_DIR / "events_2k_dataset_failures.jsonl"

assert PLAIN_PATH.exists(),    f"Missing: {PLAIN_PATH}"
assert GOLD_PATH.exists(),     f"Missing: {GOLD_PATH}"
assert FAILURES_PATH.exists(), f"Missing: {FAILURES_PATH}"
print("Dataset files found")


In [ ]:
# ====== Load dataset ======
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

plain_rows    = load_jsonl(PLAIN_PATH)
gold_rows     = load_jsonl(GOLD_PATH)
failure_rows  = load_jsonl(FAILURES_PATH)

# Exclude IDs that failed annotation quality checks
failure_ids = {r["id"] for r in failure_rows}
print(f"Excluding {len(failure_ids)} failure IDs: {sorted(failure_ids)}")

plain_by_id = {r["id"]: r for r in plain_rows if r["id"] not in failure_ids}
gold_by_id  = {r["id"]: r for r in gold_rows  if r["id"] not in failure_ids}

ids  = sorted(set(plain_by_id) & set(gold_by_id))
pool = [
    {"id": rid, "text": plain_by_id[rid]["text"], "language": plain_by_id[rid].get("language")}
    for rid in ids
]

if N_SAMPLES is not None:
    random.seed(SEED)
    pool = random.sample(pool, min(N_SAMPLES, len(pool)))
    pool = sorted(pool, key=lambda x: x["id"])

print("Rows in pool:", len(pool))
print("Example text:", pool[0]["text"][:300])


In [ ]:
# ====== Dataset analysis ======
target_counts    = Counter(gold_by_id[r["id"]].get("target")   for r in pool)
lang_counts      = Counter(gold_by_id[r["id"]].get("language") for r in pool)
events_per_row   = Counter(len(gold_by_id[r["id"]].get("events", [])) for r in pool)
broad_type_counts = Counter()
subtype_counts    = Counter()

for r in pool:
    for ev in gold_by_id[r["id"]].get("events", []):
        broad_type_counts[ev.get("event_type")] += 1
        subtype_counts[ev.get("event_subtype")] += 1

print("Targets:       ", target_counts)
print("Languages:     ", lang_counts)
print("Events per row:", events_per_row)
print("Broad types:   ", broad_type_counts)
print("Top subtypes:  ", subtype_counts.most_common(20))

# ── v3.3: dataset difficulty disclosure ──
total_slots_gold = sum(
    1 for r in gold_rows for ev in r.get("events", [])
    for k, v in ev.get("slots", {}).items()
    if v is not None and isinstance(v, str) and v.strip()
)
texts_by_id = {r["id"]: r["text"] for r in plain_rows}
unique_count = sum(
    1 for r in gold_rows for ev in r.get("events", [])
    for k, v in ev.get("slots", {}).items()
    if v is not None and isinstance(v, str) and v.strip()
    and texts_by_id.get(r["id"], "").count(v) == 1
)
print(f"\n── Dataset difficulty note ──")
print(f"Gold scalar values: {total_slots_gold}")
print(f"Appear exactly once in article: {unique_count} ({100*unique_count/total_slots_gold:.0f}%)")
print("All values are verbatim substrings of source text (0% paraphrased).")
print("Scores on this dataset are inflated relative to real newswire; "
      "report baseline lift alongside headline F1.")


In [ ]:
# ====== Canonical slot alias map + schema building ======

# Maps non-canonical slot names to their canonical form.
# Applied to BOTH gold annotations and model predictions during normalization.
SLOT_ALIASES = {
    # organizer variants
    "organiser":  "organizers",
    "organizer":  "organizers",
    # curator variants
    "curator":    "curators",
    # competition name
    "competition": "competition_name",
    # date: sports annotators used event_date and start_date interchangeably
    "event_date": "start_date",
    # winner / loser
    "winning_team": "winner",
    "losing_team":  "loser",
    # score unified
    "score": "final_score",
    # director variants
    "director": "directors",
    # key players
    "key_player": "key_players",
    # station count variants
    "num_stations":          "n_stations",
    "new_stations":          "n_stations",
    "station_count":         "n_stations",
    "num_charging_stations": "n_stations",
    "num_charging_points":   "n_stations",
    "num_sites":             "n_stations",
    # v3: book/scheme/service names — gold uses event_name for these
    "book_title":           "event_name",
    "scheme_name":          "event_name",
    "service_name":         "event_name",
    "infrastructure_name":  "event_name",
    "facility_name":        "event_name",
    # v3: road-closure reason synonym
    "cause_event":          "reason",
}

# Subtype-specific aliases (applied after global SLOT_ALIASES).
# Use when the same slot name means different things in different subtypes.
SLOT_ALIASES_BY_SUBTYPE = {
    # EVChargingRollout: gold uses n_stations for charging-point counts;
    # model often outputs fleet_size (vehicle-fleet concept from BikeShare/Ferry).
    "EVChargingRollout": {"fleet_size": "n_stations"},
}

def apply_slot_alias_for_subtype(name, subtype):
    """Apply global alias then subtype-specific override."""
    name = SLOT_ALIASES.get(name, name)
    return SLOT_ALIASES_BY_SUBTYPE.get(subtype, {}).get(name, name)

def apply_slot_alias(name):
    return SLOT_ALIASES.get(name, name)


def canonical_broad_type(x):
    if x is None: return None
    x = str(x).strip().lower()
    if x in {"sport", "sports"}:       return "sports"
    if x in {"cultural", "culture"}:   return "cultural"
    if x in {"mobility", "transport"}: return "mobility"
    return x

def normalize_subtype(x):
    if x is None: return None
    x = str(x).strip()
    return x if x else None

def infer_slot_kind(v):
    if isinstance(v, list): return "list"
    return "scalar"

# Build schema with aliases applied
subtype_to_slots_raw = defaultdict(set)
slot_kind_raw        = defaultdict(dict)
subtype_to_broad     = {}

for row in gold_rows:
    for ev in row.get("events", []):
        bt = canonical_broad_type(ev.get("event_type"))
        st = normalize_subtype(ev.get("event_subtype"))
        if st is None:
            continue
        subtype_to_broad[st] = bt
        for raw_k, v in ev.get("slots", {}).items():
            k = apply_slot_alias(raw_k)          # canonicalize
            subtype_to_slots_raw[st].add(k)
            if k not in slot_kind_raw[st]:
                slot_kind_raw[st][k] = infer_slot_kind(v)
            elif slot_kind_raw[st][k] != "list" and isinstance(v, list):
                slot_kind_raw[st][k] = "list"    # upgrade scalar -> list if list seen

subtype_to_slots = {st: sorted(slots) for st, slots in subtype_to_slots_raw.items()}
slot_kind        = {st: dict(kinds)   for st, kinds  in slot_kind_raw.items()}

# ── DS2 v4: Remove zero/near-zero coverage slots from schema ──────────────────
# Rationale: slots with ≤2 gold examples produce only hallucination FPs.
#            Slots marked (*) had fp >> gold in v3 output.
SCHEMA_EXCLUDE = {
    "Concert":               {"end_date"},                        # n=1
    "Exhibition":            {"edition", "language", "genre"},    # n=1, 1, 2
    "Festival":              {"language", "medium"},              # n=1, 1
    "Premiere":              {"event_series", "genre", "duration"},  # n=1, 1, 2
    "FootballMatch":         {"organizers", "country"},           # n=1, 1
    "RaceEvent":             {"sport_type", "home_team", "away_team"},  # n=2,2,2 (*fp=34)
    "TeamMatch":             {"sport_type"},                      # n=1 (*fp=12)
    "TransitDisruption":     {"country", "n_stations", "event_name", "reason"},  # n=1,1,1; reason: gold<5% + schema mismatch (gold=entity, model=description)
    "TransitInfrastructure": {"reason"},                          # n=2 (*fp=26)
    "TransitServiceChange":  {"contract_value", "event_name", "reason"},  # n=1,2; reason: gold sparsity (same issue as TransitDisruption)
}
for st, excl in SCHEMA_EXCLUDE.items():
    if st in subtype_to_slots:
        subtype_to_slots[st] = [s for s in subtype_to_slots[st] if s not in excl]
        for s in excl:
            slot_kind.get(st, {}).pop(s, None)
all_subtypes     = sorted(subtype_to_slots.keys())
all_broad_types  = sorted({v for v in subtype_to_broad.values() if v})

print("Broad types:", all_broad_types)
print(f"N subtypes after alias deduplication: {len(all_subtypes)}")
print("\nAmateurLeagueGame canonical slots:")
for s in subtype_to_slots.get("AmateurLeagueGame", []):
    print(f"  {s} ({slot_kind['AmateurLeagueGame'].get(s, 'scalar')})")


# ====== DS2: Consolidate 247 fine-grained gold subtypes → 10 macro subtypes ======
# Cultural: Premiere, Exhibition, Festival, Concert
# Sports:   FootballMatch, TeamMatch, RaceEvent
# Mobility: TransitDisruption, TransitInfrastructure, TransitServiceChange

GOLD_TO_MACRO = {
    # ── CULTURAL: Premiere ────────────────────────────────────────────────────
    "opera_premiere": "Premiere", "theatre_premiere": "Premiere",
    "orchestral_premiere": "Premiere", "orchestral_concert_premiere": "Premiere",
    "concert_premiere": "Premiere", "symphony_premiere": "Premiere",
    "gala_premiere_theatre": "Premiere", "film_premiere": "Premiere",
    "premiere_musical_work": "Premiere", "arts_festival_premiere": "Premiere",
    "festival_premiere": "Premiere", "music_festival_premiere": "Premiere",
    "open_air_concert_premiere": "Premiere", "choral_symphony_premiere": "Premiere",
    "theatre_season_premiere": "Premiere", "orchestral_concert_premiere": "Premiere",
    # ── CULTURAL: Exhibition ──────────────────────────────────────────────────
    "exhibition_opening": "Exhibition", "exhibition_retrospective": "Exhibition",
    "sculpture_retrospective_exhibition": "Exhibition",
    "sculpture_retrospective": "Exhibition", "sculpture_exhibition": "Exhibition",
    "photography_exhibition_opening": "Exhibition",
    "photography_exhibition": "Exhibition",
    "photography_retrospective_festival": "Exhibition",
    "photography_festival_opening": "Exhibition",
    "group_exhibition_opening": "Exhibition",
    "exhibition_opening_retrospective": "Exhibition",
    "film_retrospective_screening": "Exhibition",
    "film_retrospective": "Exhibition", "film_screening_retrospective": "Exhibition",
    "film_festival_opening": "Exhibition",
    # ── CULTURAL: Festival ────────────────────────────────────────────────────
    "festival_opening": "Festival", "arts_festival_opening": "Festival",
    "music_festival_opening": "Festival", "theatre_festival_opening": "Festival",
    "performing_arts_festival_opening": "Festival",
    "book_festival_opening": "Festival", "dance_festival_opening": "Festival",
    "jazz_festival_opening": "Festival", "theatre_festival": "Festival",
    "music_festival": "Festival", "performing_arts_festival": "Festival",
    "arts_festival": "Festival", "flamenco_festival": "Festival",
    "festival_opening_concert": "Festival", "festival_opening_gala": "Festival",
    "festival_edition_opening": "Festival",
    "book_fair_opening": "Festival", "book_fair": "Festival",
    "art_fair_opening": "Festival",
    "book_fair_opening": "Festival", "book_fair": "Festival",
    # ── CULTURAL: Concert ─────────────────────────────────────────────────────
    "concert": "Concert", "orchestral_concert": "Concert",
    "concert_retrospective": "Concert", "concert_series_opening": "Concert",
    "concert_season_opening": "Concert", "concert_retrospective_series": "Concert",
    "concert_series_retrospective": "Concert", "concert_series": "Concert",
    "orchestral_concert_season_opening": "Concert",
    "orchestral_retrospective": "Concert", "orchestral_residency": "Concert",
    "open_air_concert": "Concert", "vocal_recital": "Concert",
    "piano_recital": "Concert", "tribute_concert": "Concert",
    "retrospective_concert": "Concert", "retrospective_concert_screening": "Concert",
    "chamber_music_retrospective": "Concert", "concert_double_bill": "Concert",
    "concert_retrospective_series": "Concert",
    "music_retrospective_festival": "Concert",
    # ── SPORTS: FootballMatch ─────────────────────────────────────────────────
    "football_match": "FootballMatch", "football_league_match": "FootballMatch",
    "football_derby": "FootballMatch", "football_match_derby": "FootballMatch",
    "football_derby_match": "FootballMatch", "football_match_draw": "FootballMatch",
    "football_match_result": "FootballMatch",
    "football_match_comeback_victory": "FootballMatch",
    "football_cup_match": "FootballMatch", "football_match_preview": "FootballMatch",
    "football_match_europa_league": "FootballMatch",
    "football_match_champions_league_semifinal": "FootballMatch",
    "football_league_title_decider": "FootballMatch",
    "football_league_match_title_clincher": "FootballMatch",
    "football_league_match_draw": "FootballMatch",
    "football_league_derby": "FootballMatch", "football_knockout_match": "FootballMatch",
    "football_derby_draw": "FootballMatch",
    "football_playoff_qualifier": "FootballMatch",
    "football_match_relegation_playoff": "FootballMatch",
    "cup_match": "FootballMatch", "cup_match_result": "FootballMatch",
    "cup_semifinal": "FootballMatch", "cup_semifinal_match": "FootballMatch",
    "cup_semi_final": "FootballMatch", "cup_quarterfinal": "FootballMatch",
    "cup_match_semifinal": "FootballMatch",
    "cup_match_quarterfinal": "FootballMatch",
    "cup_match_elimination": "FootballMatch", "cup_match_comeback": "FootballMatch",
    "cup_final": "FootballMatch", "cup_knockout_match": "FootballMatch",
    "league_match_draw": "FootballMatch", "league_title_decider": "FootballMatch",
    "league_football_match": "FootballMatch", "league_derby": "FootballMatch",
    "local_derby": "FootballMatch", "championship_final": "FootballMatch",
    "gaelic_football_match": "FootballMatch",
    # ── SPORTS: TeamMatch ─────────────────────────────────────────────────────
    "rugby_union_match": "TeamMatch", "rugby_union_knockout_match": "TeamMatch",
    "rugby_union_international_match": "TeamMatch",
    "handball_league_match": "TeamMatch", "handball_match_semifinal": "TeamMatch",
    "handball_match": "TeamMatch", "ice_hockey_match": "TeamMatch",
    "ice_hockey_semifinal": "TeamMatch", "ice_hockey_playoff_final": "TeamMatch",
    "ice_hockey_league_match": "TeamMatch",
    "basketball_league_match": "TeamMatch",
    "volleyball_match": "TeamMatch", "volleyball_semifinal": "TeamMatch",
    "water_polo_cup_final": "TeamMatch", "fencing_semifinal": "TeamMatch",
    "fencing_championship": "TeamMatch",
    # ── SPORTS: RaceEvent ─────────────────────────────────────────────────────
    "marathon_race": "RaceEvent", "road_race_result": "RaceEvent",
    "road_race_finish": "RaceEvent", "road_race_championship": "RaceEvent",
    "cycling_stage_race_opening_stage": "RaceEvent",
    "cycling_stage_race_finale": "RaceEvent", "cycling_stage_finish": "RaceEvent",
    "track_cycling_semifinal": "RaceEvent",
    "athletics_track_final": "RaceEvent", "athletics_race_finish": "RaceEvent",
    "athletics_championship_race": "RaceEvent",
    "tennis_match_result": "RaceEvent", "tennis_match_victory": "RaceEvent",
    "tennis_final": "RaceEvent", "tennis_tie_qualifier": "RaceEvent",
    "golf_tournament_victory": "RaceEvent",
    "rowing_championship": "RaceEvent", "regatta_race_result": "RaceEvent",
    "regatta_final": "RaceEvent", "canoe_sprint_race": "RaceEvent",
    # ── MOBILITY: TransitDisruption ───────────────────────────────────────────
    "transit_disruption_signalling_fault": "TransitDisruption",
    "transit_disruption_signal_failure": "TransitDisruption",
    "transit_disruption_signal_fault": "TransitDisruption",
    "transit_disruption_signalling_failure": "TransitDisruption",
    "transit_disruption_line_closure": "TransitDisruption",
    "transit_disruption_track_closure": "TransitDisruption",
    "transit_disruption_track_works": "TransitDisruption",
    "transit_disruption_points_failure": "TransitDisruption",
    "transit_disruption_engineering_works": "TransitDisruption",
    "transit_disruption_suspension": "TransitDisruption",
    "transit_disruption_line_suspension": "TransitDisruption",
    "transit_disruption_infrastructure_fault": "TransitDisruption",
    "transit_disruption": "TransitDisruption",
    "transit_disruption_track_replacement": "TransitDisruption",
    "transit_disruption_track_maintenance": "TransitDisruption",
    "transit_disruption_track_damage": "TransitDisruption",
    "transit_disruption_technical_fault": "TransitDisruption",
    "transit_disruption_route_suspension": "TransitDisruption",
    "transit_disruption_rolling_stock_withdrawal": "TransitDisruption",
    "transit_disruption_rail_replacement": "TransitDisruption",
    "transit_disruption_planned_engineering": "TransitDisruption",
    "transit_disruption_planned_closure": "TransitDisruption",
    "transit_disruption_partial_suspension": "TransitDisruption",
    "transit_disruption_partial_closure": "TransitDisruption",
    "transit_disruption_overhead_wire": "TransitDisruption",
    "transit_disruption_overhead_line_failure": "TransitDisruption",
    "transit_disruption_overhead_cable": "TransitDisruption",
    "transit_disruption_landslip": "TransitDisruption",
    "transit_disruption_infrastructure_works": "TransitDisruption",
    "transit_disruption_gleisbau": "TransitDisruption",
    "transit_disruption_event_traffic_management": "TransitDisruption",
    "transit_disruption_derailment": "TransitDisruption",
    "transit_disruption_service_suspension": "TransitDisruption",
    "transit_disruption_rail_closure": "TransitDisruption",
    "transit_suspension": "TransitDisruption",
    "transit_line_closure": "TransitDisruption",
    "transit_line_suspension": "TransitDisruption",
    "transit_partial_suspension": "TransitDisruption",
    "transit_strike": "TransitDisruption",
    "rail_service_suspension": "TransitDisruption",
    "rail_service_suspension_signalling_fault": "TransitDisruption",
    "rail_suspension": "TransitDisruption", "rail_line_closure": "TransitDisruption",
    "tram_suspension": "TransitDisruption", "tram_service_suspension": "TransitDisruption",
    "tram_line_closure": "TransitDisruption", "track_closure": "TransitDisruption",
    "station_closure": "TransitDisruption",
    "station_closure_track_repair": "TransitDisruption",
    "bus_route_suspension": "TransitDisruption",
    "bus_service_suspension": "TransitDisruption",
    "bus_driver_strike": "TransitDisruption",
    "metro_service_suspension": "TransitDisruption",
    "metro_service_disruption": "TransitDisruption",
    "metro_signalling_disruption": "TransitDisruption",
    "line_closure_track_works": "TransitDisruption",
    "line_closure_infrastructure": "TransitDisruption",
    # ── MOBILITY: TransitInfrastructure ───────────────────────────────────────
    "metro_line_extension": "TransitInfrastructure",
    "metro_line_extension_project": "TransitInfrastructure",
    "metro_line_extension_groundbreaking": "TransitInfrastructure",
    "metro_line_extension_groundwork": "TransitInfrastructure",
    "metro_line_extension_construction_start": "TransitInfrastructure",
    "metro_line_extension_construction": "TransitInfrastructure",
    "metro_line_extension_approved": "TransitInfrastructure",
    "metro_line_expansion_groundbreaking": "TransitInfrastructure",
    "metro_extension_with_construction_closures": "TransitInfrastructure",
    "metro_extension_corridor_closure": "TransitInfrastructure",
    "underground_line_extension_groundbreaking": "TransitInfrastructure",
    "tram_line_extension_construction_start": "TransitInfrastructure",
    "tram_line_extension_approval": "TransitInfrastructure",
    "tram_line_construction_approval": "TransitInfrastructure",
    "transit_line_expansion_groundwork": "TransitInfrastructure",
    "transit_suspension_engineering_works": "TransitInfrastructure",
    "transit_suspension_track_replacement": "TransitInfrastructure",
    "transit_suspension_track_repairs": "TransitInfrastructure",
    "transit_suspension_infrastructure": "TransitInfrastructure",
    "transit_closure_infrastructure_works": "TransitInfrastructure",
    "service_suspension_engineering_works": "TransitInfrastructure",
    "service_suspension_infrastructure_works": "TransitInfrastructure",
    "service_suspension_infrastructure_failure": "TransitInfrastructure",
    "infrastructure_project_approval": "TransitInfrastructure",
    "infrastructure_project_with_service_disruption": "TransitInfrastructure",
    "line_opening": "TransitInfrastructure",
    "line_opening_with_construction_disruption": "TransitInfrastructure",
    "planned_transit_disruption": "TransitInfrastructure",
    "planned_line_closure": "TransitInfrastructure",
    # ── MOBILITY: TransitServiceChange ────────────────────────────────────────
    "fare_change": "TransitServiceChange",
    "transit_capacity_increase": "TransitServiceChange",
    "transit_frequency_increase": "TransitServiceChange",
    "transit_service_reinforcement": "TransitServiceChange",
    "special_service_match_day": "TransitServiceChange",
    "special_service_frequency_increase": "TransitServiceChange",
    "special_service_extension": "TransitServiceChange",
    "special_service_deployment": "TransitServiceChange",
    "event_related_transit_reinforcement": "TransitServiceChange",
    "event_driven_transit_reinforcement": "TransitServiceChange",
    # ── 2k v5: new cultural subtypes ─────────────────────────────────────────
    # Concert
    "anniversary_concert":              "Concert",
    "chamber_concert":                  "Concert",
    "chamber_music_residency":          "Concert",
    "choral_symphony_world_premiere":   "Premiere",
    "classical_concert":                "Concert",
    "concert_gala_opening":             "Concert",
    "concert_liederabend":              "Concert",
    "concert_opening":                  "Concert",
    "concert_residency":                "Concert",
    "festival_closing_concert":         "Concert",
    "festival_concert":                 "Concert",
    "jazz_concert_residency":           "Concert",
    "open_air_orchestral_concert":      "Concert",
    "opera_performance":                "Concert",
    "orchestral_concert_season_opener": "Concert",
    "orchestral_concert_series":        "Concert",
    "orchestral_gala_opening":          "Concert",
    "orchestral_retrospective_concert": "Concert",
    "orchestral_world_premiere":        "Premiere",
    "outdoor_concert":                  "Concert",
    "retrospective_concert_series":     "Concert",
    # Exhibition
    "art_exhibition_opening":           "Exhibition",
    "art_installation_premiere":        "Exhibition",
    "exhibition_premiere":              "Exhibition",
    "group_exhibition":                 "Exhibition",
    "literary_venue_opening":           "Exhibition",
    "multimedia_installation_premiere": "Exhibition",
    # Festival
    "art_fair":                         "Festival",
    "art_fair_book_fair_opening":       "Festival",
    "arts_festival_launch":             "Festival",
    "book_fair_launch":                 "Festival",
    "book_fair_premiere":               "Festival",
    "book_launch":                      "Festival",
    "book_launch_opening_reading":      "Festival",
    "chamber_music_festival":           "Festival",
    "chamber_music_festival_opening":   "Festival",
    "dance_festival":                   "Festival",
    "festival":                         "Festival",
    "festival_announcement":            "Festival",
    "festival_inaugural_edition":       "Festival",
    "festival_opening_night":           "Festival",
    "festival_opening_performance":     "Festival",
    "festival_premiere_announcement":   "Festival",
    "festival_programme_announcement":  "Festival",
    "festival_programme_expansion":     "Festival",
    "festival_season_opening":          "Festival",
    "film_festival":                    "Festival",
    "interdisciplinary_festival_opening": "Festival",
    "jazz_festival":                    "Festival",
    "light_art_festival_opening":       "Festival",
    "literature_festival":              "Festival",
    "open_air_painting_festival_opening": "Festival",
    "theatre_opening_festival":         "Festival",
    "visual_art_festival":              "Festival",
    "visual_art_festival_opening":      "Festival",
    # Premiere
    "dance_premiere":                   "Premiere",
    "film_festival_premiere":           "Premiere",
    "playwright_residency_launch":      "Premiere",
    "programme_preview_presentation":   "Premiere",
    "theatre_festival_premiere":        "Premiere",
    "theatre_premiere_opening_night":   "Premiere",
    "theatre_reopening_festival_premiere": "Premiere",
    # ── 2k v5: new sports subtypes ───────────────────────────────────────────
    # FootballMatch
    "Gaelic_football_championship_final":  "FootballMatch",
    "cup_match_knockout":                  "FootballMatch",
    "cup_replay":                          "FootballMatch",
    "cup_semifinal_football":              "FootballMatch",
    "football_group_stage_match":          "FootballMatch",
    "football_international_group_match":  "FootballMatch",
    "football_match_knockout":             "FootballMatch",
    "football_match_qualifier":            "FootballMatch",
    "football_match_relegation":           "FootballMatch",
    "football_match_semifinal":            "FootballMatch",
    "football_regional_derby":             "FootballMatch",
    "football_relegation_match":           "FootballMatch",
    "football_world_cup_qualifier":        "FootballMatch",
    "futsal_league_match":                 "FootballMatch",
    "international_football_match":        "FootballMatch",
    "international_qualifier_match":       "FootballMatch",
    "league_match_title_clincher":         "FootballMatch",
    "league_season_opener":                "FootballMatch",
    "regional_league_football_match":      "FootballMatch",
    "relegation_decider_match":            "FootballMatch",
    "relegation_playoff":                  "FootballMatch",
    "women_football_derby":                "FootballMatch",
    "youth_football_match":                "FootballMatch",
    # TeamMatch
    "Australian rules football match":     "TeamMatch",
    "basketball_championship_final":       "TeamMatch",
    "basketball_game":                     "TeamMatch",
    "basketball_match":                    "TeamMatch",
    "fencing_championship_final":          "TeamMatch",
    "fencing_match_semifinal":             "TeamMatch",
    "golf_team_match":                     "TeamMatch",
    "handball_derby_draw":                 "TeamMatch",
    "handball_semifinal":                  "TeamMatch",
    "ice_hockey_championship_final":       "TeamMatch",
    "league_match":                        "TeamMatch",
    "rugby_international_match":           "TeamMatch",
    "rugby_union_derby_match":             "TeamMatch",
    "rugby_union_qualifier":               "TeamMatch",
    "semifinal_match":                     "TeamMatch",
    "squash_match_semifinal":              "TeamMatch",
    "volleyball_match_semifinal":          "TeamMatch",
    "water_polo_match":                    "TeamMatch",
    "water_polo_semifinal":                "TeamMatch",
    "wrestling_championship":              "TeamMatch",
    "wrestling_quarterfinal_match":        "TeamMatch",
    "wrestling_semifinal":                 "TeamMatch",
    # RaceEvent
    "cycling_stage_race_finish":           "RaceEvent",
    "cycling_stage_race_opening":          "RaceEvent",
    "marathon_finish":                     "RaceEvent",
    "open-water swimming race":            "RaceEvent",
    "regatta_race":                        "RaceEvent",
    "regatta_race_final":                  "RaceEvent",
    "regatta_semifinal":                   "RaceEvent",
    "rowing_race":                         "RaceEvent",
    "rowing_regatta_race":                 "RaceEvent",
    "speed_skating_championship_final":    "RaceEvent",
    "swimming_championship_final":         "RaceEvent",
    "swimming_championship_race":          "RaceEvent",
    "swimming_relay_semifinal":            "RaceEvent",
    "tennis_match":                        "RaceEvent",
    "tennis_match_comeback_victory":       "RaceEvent",
    "tennis_match_quarterfinal":           "RaceEvent",
    "tennis_tie":                          "RaceEvent",
    "tennis_tournament_match":             "RaceEvent",
    "track_and_field_final":               "RaceEvent",
    # ── 2k v5: new mobility subtypes ─────────────────────────────────────────
    # TransitDisruption
    "bus_route_diversion_during_construction": "TransitDisruption",
    "bus_route_suspension_roadworks":      "TransitDisruption",
    "bus_service_disruption":              "TransitDisruption",
    "bus_service_suspension_resurfacing":  "TransitDisruption",
    "bus_suspension_race_closure":         "TransitDisruption",
    "emergency_track_closure":             "TransitDisruption",
    "infrastructure_works_disruption":     "TransitDisruption",
    "line_closure":                        "TransitDisruption",
    "line_closure_engineering_works":      "TransitDisruption",
    "line_closure_infrastructure_works":   "TransitDisruption",
    "metro_line_closure":                  "TransitDisruption",
    "rail_line_suspension":                "TransitDisruption",
    "rail_service_disruption":             "TransitDisruption",
    "rail_strike":                         "TransitDisruption",
    "road_closure_bus_suspension":         "TransitDisruption",
    "road_closure_transit_suspension":     "TransitDisruption",
    "road_closure_water_main_rupture":     "TransitDisruption",
    "road_rail_closure":                   "TransitDisruption",
    "station_closure_with_replacement_service": "TransitDisruption",
    "station_closure_with_rerouting":      "TransitDisruption",
    "strike_averted":                      "TransitDisruption",
    "transit_closure_emergency_repair":    "TransitDisruption",
    "transit_closure_maintenance":         "TransitDisruption",
    "transit_disruption_closure":          "TransitDisruption",
    "transit_disruption_construction":     "TransitDisruption",
    "transit_disruption_crowd_event":      "TransitDisruption",
    "transit_disruption_crowd_management": "TransitDisruption",
    "transit_disruption_event_related":    "TransitDisruption",
    "transit_disruption_fire":             "TransitDisruption",
    "transit_disruption_infrastructure":   "TransitDisruption",
    "transit_disruption_infrastructure_damage": "TransitDisruption",
    "transit_disruption_infrastructure_failure": "TransitDisruption",
    "transit_disruption_landslide":        "TransitDisruption",
    "transit_disruption_maintenance":      "TransitDisruption",
    "transit_disruption_match_day":        "TransitDisruption",
    "transit_disruption_overhead_wire_failure": "TransitDisruption",
    "transit_disruption_rail":             "TransitDisruption",
    "transit_disruption_resolved":         "TransitDisruption",
    "transit_disruption_road_closure":     "TransitDisruption",
    "transit_disruption_station_closure":  "TransitDisruption",
    "transit_disruption_timetable_reduction": "TransitDisruption",
    "transit_disruption_track_fault":      "TransitDisruption",
    "transit_disruption_track_repair":     "TransitDisruption",
    "transit_disruption_trackwork":        "TransitDisruption",
    "transit_disruption_tram":             "TransitDisruption",
    "transit_disruption_tunnel_closure":   "TransitDisruption",
    "transit_diversion":                   "TransitDisruption",
    "transit_partial_closure":             "TransitDisruption",
    "transit_route_suspension":            "TransitDisruption",
    "transit_service_restriction_trackwork": "TransitDisruption",
    "transit_service_suspension":          "TransitDisruption",
    "transit_suspension_infrastructure_failure": "TransitDisruption",
    "transit_suspension_maintenance":      "TransitDisruption",
    "transit_suspension_partial":          "TransitDisruption",
    "transit_suspension_signalling_failure": "TransitDisruption",
    "transit_suspension_signalling_fault": "TransitDisruption",
    "tunnel_closure_maintenance":          "TransitDisruption",
    # TransitInfrastructure
    "infrastructure_construction_route_closure": "TransitInfrastructure",
    "light_rail_extension_approved":       "TransitInfrastructure",
    "line_closure_renovation":             "TransitInfrastructure",
    "line_expansion_station_opening":      "TransitInfrastructure",
    "line_extension":                      "TransitInfrastructure",
    "line_opening_international_night_train": "TransitInfrastructure",
    "metro_expansion_infrastructure_project": "TransitInfrastructure",
    "metro_extension_construction_disruption": "TransitInfrastructure",
    "metro_extension_construction_start":  "TransitInfrastructure",
    "metro_line_expansion":                "TransitInfrastructure",
    "metro_line_expansion_project":        "TransitInfrastructure",
    "metro_line_infrastructure_project":   "TransitInfrastructure",
    "planned_service_suspension":          "TransitInfrastructure",
    "rail_infrastructure_project":         "TransitInfrastructure",
    "rail_infrastructure_project_launch":  "TransitInfrastructure",
    "rail_line_extension":                 "TransitInfrastructure",
    "rail_line_extension_approved":        "TransitInfrastructure",
    "service_expansion":                   "TransitInfrastructure",
    "service_extension":                   "TransitInfrastructure",
    "track_closure_infrastructure_works":  "TransitInfrastructure",
    "tram_line_construction_project":      "TransitInfrastructure",
    "tram_line_extension_approved":        "TransitInfrastructure",
    "tram_line_extension_project":         "TransitInfrastructure",
    "tram_service_suspension_track_renewal": "TransitInfrastructure",
    "transit_disruption_track_renewal":    "TransitInfrastructure",
    "transit_infrastructure_approval":     "TransitInfrastructure",
    "transit_infrastructure_expansion":    "TransitInfrastructure",
    "transit_infrastructure_project":      "TransitInfrastructure",
    "transit_line_extension":              "TransitInfrastructure",
    "transit_suspension_infrastructure_repair": "TransitInfrastructure",
    "transit_suspension_infrastructure_works": "TransitInfrastructure",
    # TransitServiceChange
    "road_closure":                            "TransitServiceChange",
    "road_closure_and_transit_reinforcement":  "TransitServiceChange",
    "road_closure_and_transit_surge":          "TransitServiceChange",
    "road_closure_bus_diversion":              "TransitServiceChange",
    "service_reinforcement":                   "TransitServiceChange",
    "transit_capacity_increase_and_road_closure": "TransitServiceChange",
    "transit_disruption_special_service":      "TransitServiceChange",
    "transit_frequency_increase_and_road_closure": "TransitServiceChange",
    "transit_service_extension":               "TransitServiceChange",
    "transit_service_reduction":               "TransitServiceChange",
    "transit_special_service":                 "TransitServiceChange",
    "tram_suspension_event_related":           "TransitServiceChange",
    "emergency_shuttle_deployment": "TransitServiceChange",
    "road_closure_event_induced": "TransitServiceChange",
    "road_closure_and_transit_extension": "TransitServiceChange",
    "road_closure_and_shuttle_service": "TransitServiceChange",
    "road_closure_and_bus_suspension": "TransitServiceChange",
    "bus_route_diversion": "TransitServiceChange",
    "service_suspension": "TransitServiceChange",
    "transit_disruption_event_traffic_management": "TransitServiceChange",
}

def macro_subtype(st):
    return GOLD_TO_MACRO.get(st, st)

# Remap schema to macro subtypes
from collections import defaultdict as _dd
_new_st_slots = _dd(set)
_new_slot_kind = _dd(dict)
_new_broad = {}
for st, slots in subtype_to_slots.items():
    mst = macro_subtype(st)
    _new_broad[mst] = subtype_to_broad.get(st, "cultural")
    for sl in slots:
        _new_st_slots[mst].add(sl)
    for sl, k in slot_kind.get(st, {}).items():
        ex = _new_slot_kind[mst].get(sl)
        _new_slot_kind[mst][sl] = "list" if (k == "list" or ex == "list") else "scalar"

subtype_to_slots = {st: sorted(s) for st, s in _new_st_slots.items()}
slot_kind        = {st: dict(k) for st, k in _new_slot_kind.items()}
subtype_to_broad = dict(_new_broad)
# v5.1 FIX: SCHEMA_EXCLUDE is keyed by MACRO subtype names, so it must be
# applied AFTER the macro-remap. The earlier application (pre-remap, while the
# schema was still keyed by fine-grained names) was a silent no-op.
for _st, _excl in SCHEMA_EXCLUDE.items():
    if _st in subtype_to_slots:
        subtype_to_slots[_st] = [s for s in subtype_to_slots[_st] if s not in _excl]
        for _s in _excl:
            slot_kind.get(_st, {}).pop(_s, None)
all_subtypes     = sorted(subtype_to_slots.keys())

# Remap gold events in-place so evaluation uses macro subtypes
for _row in gold_rows:
    for _ev in _row.get("events", []):
        _st = _ev.get("event_subtype", "")
        if _st in GOLD_TO_MACRO:
            _ev["event_subtype"] = GOLD_TO_MACRO[_st]
        # Also drop politics events from gold
        if _ev.get("event_type") == "politics":
            _ev["_skip"] = True

# Filter politics events out of gold_rows
for _row in gold_rows:
    _row["events"] = [e for e in _row.get("events", []) if not e.get("_skip")]

print(f"Macro subtypes: {all_subtypes}")
print(f"Broad types: {sorted(set(subtype_to_broad.values()))}")

In [ ]:
# ====== Schema text — list/scalar hints, grouped by domain ======
# v3: one-line hints to distinguish similar subtypes
SUBTYPE_HINTS = {
    # Cultural
    "Premiere":               "world or local premiere of a new work — opera, theatre, orchestral, film",
    "Exhibition":             "visual art display open for multiple days — includes retrospectives and photography shows",
    "Festival":               "multi-day cultural event — includes book fairs, art fairs, music and theatre festivals",
    "Concert":                "single-night live music performance — orchestral, jazz, recital, open-air",
    # Sports
    "FootballMatch":          "any football (soccer) match — league, cup, derby, knockout, draw, or final",
    "TeamMatch":              "non-football team sport — rugby, handball, ice hockey, basketball, volleyball, water polo, fencing",
    "RaceEvent":              "individual racing or athletics — marathon, road race, cycling stage, tennis, rowing, canoe",
    # Mobility / street events
    "TransitDisruption":      "unplanned service disruption — signal fault, suspension, derailment, strike, track closure",
    "TransitInfrastructure":  "planned construction or extension — metro/tram line groundbreaking, new station approval",
    "TransitServiceChange":   "planned service change — fare change, extra services, route diversion, frequency increase",
}

def build_schema_text():
    lines = ["Allowed event_type values: cultural, sports, mobility", ""]
    for domain in ["cultural", "sports", "mobility"]:
        domain_subtypes = [st for st in all_subtypes if subtype_to_broad.get(st) == domain]
        if not domain_subtypes:
            continue
        lines.append(f"── {domain.upper()} subtypes ──")
        for st in domain_subtypes:
            parts = []
            for s in subtype_to_slots[st]:
                kind = slot_kind[st].get(s, "scalar")
                parts.append(f"{s}[]" if kind == "list" else s)
            hint = SUBTYPE_HINTS.get(st, "")
            hint_str = f"  # {hint}" if hint else ""
            lines.append(f"  {st}{hint_str}: {', '.join(parts)}")
        lines.append("")
    return "\n".join(lines).strip()

SCHEMA_TEXT = build_schema_text()
print(SCHEMA_TEXT[:3000])


In [ ]:
# ====== Few-shot examples (synthetic, no data leakage) ======
FEW_SHOT = """
--- EXAMPLE 1: cultural, Festival (book fair) ---
Article:
The Stadtbibliothek in Erfurt hosted the 12th annual Lesefest Thüringen on Saturday,
a three-day literary festival featuring readings by novelists Greta Sommer and Paul Naumann.
Around 3,400 visitors attended, with day passes priced at €9.

Output:
{"events": [{"event_type": "cultural", "event_subtype": "Festival", "slots": {
  "event_name": "Lesefest Thüringen", "venue": "Stadtbibliothek",
  "city": "Erfurt", "start_date": "Saturday", "edition": "12",
  "actual_attendance": "3400", "ticket_price": "€9",
  "performers": ["Greta Sommer", "Paul Naumann"]
}}]}

--- EXAMPLE 2: sports, FootballMatch (no city stated -> 'city' omitted) ---
Article:
FC Riverside beat Athletic Northdale 2–1 in the Northern League Cup at Millbrook Stadium
on Sunday before a crowd of 4,800. Key contributions came from striker Lena Brandt.

Output:
{"events": [{"event_type": "sports", "event_subtype": "FootballMatch", "slots": {
  "home_team": "FC Riverside", "away_team": "Athletic Northdale",
  "final_score": "2-1", "winner": "FC Riverside", "loser": "Athletic Northdale",
  "venue": "Millbrook Stadium",
  "competition_name": "Northern League Cup",
  "start_date": "Sunday", "attendance": "4800",
  "key_players": ["Lena Brandt"]
}}]}

--- EXAMPLE 3: no target event — policy discussion ---
Article:
The city council discussed plans to renovate the central library next year.

Output:
{"events": []}

--- EXAMPLE 4: no target event — statistics / survey report ---
Article:
A national infrastructure survey published last week found that dedicated cycle lanes across
British cities had grown by eleven percent between 2019 and 2024. The report, compiled from
municipal data, cited funding shortfalls in smaller towns as the main obstacle to faster
expansion but made no announcement of new schemes or launch dates.

Output:
{"events": []}

--- EXAMPLE 5: mobility, TransitDisruption ---
Article:
A signalling fault near the Hauptbahnhof in Potsdam brought Stadtwerke Potsdam's tram network
to a standstill on Friday morning, suspending Linie 4 und Linie 9 for around three hours
and affecting an estimated 12,000 passengers. Services were fully restored by noon.

Output:
{"events": [{"event_type": "mobility", "event_subtype": "TransitDisruption", "slots": {
  "city": "Potsdam", "operator": "Stadtwerke Potsdam",
  "affected_lines": ["Linie 4", "Linie 9"],
  "reason": "signalling fault near the Hauptbahnhof",
  "start_date": "Friday morning", "duration": "around three hours",
  "estimated_affected_passengers": "12000", "ongoing": "false"
}}]}

--- EXAMPLE 6: cultural, Premiere (new orchestral work) ---
Article:
The Nationaltheater Weimar staged the world premiere of composer Lena Brandmeyer's
orchestral work "Im Zwischenraum" on Friday evening before an audience of 320.
The performance featured violinist Marco Stelzer and the Weimar Chamber Orchestra,
conducted by Goran Petrak.

Output:
{"events": [{"event_type": "cultural", "event_subtype": "Premiere", "slots": {
  "event_name": "Im Zwischenraum", "venue": "Nationaltheater Weimar", "city": "Weimar",
  "start_date": "Friday evening", "actual_attendance": "320",
  "artists": ["Lena Brandmeyer"],
  "directors": ["Goran Petrak"],
  "performers": ["Marco Stelzer", "Weimar Chamber Orchestra"]
}}]}

--- EXAMPLE 7: sports, TeamMatch (no city stated -> 'city' omitted) ---
Article:
Flensburg Handewitt edged THW Kiel 29–28 in the Handball-Bundesliga at the
Sparkassen Arena on Saturday before 6,200 fans. Jan-Erik Dahl scored the decisive
late goal to secure the home win.

Output:
{"events": [{"event_type": "sports", "event_subtype": "TeamMatch", "slots": {
  "home_team": "Flensburg Handewitt", "away_team": "THW Kiel",
  "winner": "Flensburg Handewitt", "loser": "THW Kiel",
  "final_score": "29-28", "competition_name": "Handball-Bundesliga",
  "venue": "Sparkassen Arena",
  "start_date": "Saturday", "attendance": "6200",
  "key_players": ["Jan-Erik Dahl"]
}}]}
""".strip()

In [ ]:
# ====== Prompt builder (tightened anti-over-extraction rules) ======
RULES = """You are an information extraction system.

Task: Extract zero or more events from the article.

Return exactly one JSON object:
{"events": [{"event_type": "...", "event_subtype": "...", "slots": {...}}, ...]}
If no target events are present: {"events": []}

CORE PRINCIPLE - extract ONLY what is explicitly written:
1. Copy slot values EXACTLY from the article (names, venues, dates, scores). Do not paraphrase, translate, or reformat - except the numeric normalization below.
2. Never infer, derive, compute, or guess. If a fact is not stated in words, OMIT the slot.
3. Never output "null", "none", "unknown", "n/a", "not specified", "keine Angabe", "-", or "". Omit the slot instead.
4. Use only the slot names listed for the chosen subtype; omit every slot not mentioned.
5. event_type must be one of: cultural, sports, mobility. event_subtype must be from the allowed list.
6. Output valid JSON only. No markdown, no explanation.

AVOID OVER-EXTRACTION (these are the most frequent mistakes - be strict):
7. city: fill ONLY when a city or town is named in the text. NEVER derive it from a venue, stadium, arena, team, or operator name, and never substitute a country or region (e.g. "Austria", "Merseyside"). If no settlement is named in words, omit city.
8. duration: fill ONLY when an explicit duration phrase is written (e.g. "for three hours", "an 11-day closure"). Never calculate it from start/end dates.
9. final_score: fill ONLY when an explicit score is written; copy it verbatim. Do not assemble a score from separate numbers.
10. list slots (organizers[], performers[], artists[], key_players[], ...): add a name ONLY if the text explicitly gives it that role. Do not promote a merely-mentioned person or organisation into a role.

BE COMPLETE (avoid misses):
11. In list slots, include EVERY item explicitly mentioned for that role - especially each line in affected_lines[] and each route in affected_routes[].

NUMERIC NORMALIZATION (numeric slots only):
12. Words to digits ("seventh" -> "7"); drop thousands separators ("1,200" -> "1200"); take the number from a phrase ("around 800 visitors" -> "800"). Leave names, dates, venues, scores, prices unchanged.

MOBILITY:
13. Use 'city' for the city; 'affected_lines'[] for line numbers/names ("S3", "Linie 7"); 'affected_routes'[] for route names ("Route 12", "N7").
14. ongoing: set true ONLY if the text says the event is in progress at the time of publication. Announcements, approvals, groundbreakings, and planned or completed works are NOT ongoing - omit or set false.

NO-EVENT - return {"events": []} when the article is:
15. a statistics/trend report ("fell X% over N years"), a historical retrospective, speculation about an event with no confirmed date/venue, or a policy debate with no concrete decision. A real event is a specific named occurrence with at least one concrete detail (venue, date, or participant).

EVENT COUNT:
16. Extract only events that actually occur as the primary subject. Ignore events mentioned incidentally, historically, or for comparison. When in doubt, prefer fewer events."""

def build_prompt(text: str) -> str:
    article = text if MAX_INPUT_CHARS is None else text[:MAX_INPUT_CHARS]
    user_content = (
        f"{RULES}\n\n"
        f"SCHEMA:\n{SCHEMA_TEXT}\n\n"
        f"EXAMPLES:\n{FEW_SHOT}\n\n"
        f"---\nArticle:\n{article}\n\nOutput:"
    )
    msgs = [
        {"role": "system",  "content": "You are a precise information extraction system that outputs only valid JSON."},
        {"role": "user",    "content": user_content},
    ]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


In [ ]:
# ====== Load model + tokenizer ======
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

def load_model_tokenizer(model_name, use_4bit=True, token=""):
    tok = AutoTokenizer.from_pretrained(model_name, use_fast=True, token=token or None)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "left"

    kwargs = dict(device_map="auto", dtype=torch.float16, token=token or None)
    if use_4bit:
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
        )
    mdl = AutoModelForCausalLM.from_pretrained(model_name, **kwargs)
    mdl.eval()
    return mdl, tok

model, tokenizer = load_model_tokenizer(MODEL_NAME, USE_4BIT, HF_TOKEN)
print("Loaded:", MODEL_NAME, "| 4-bit:", USE_4BIT)


In [ ]:
# ====== Seed utility ======
import random
import numpy as np

def set_seed(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

# ====== Batched generation ======
# Key fix: output extraction uses padded_input_len (inputs["input_ids"].shape[1])
# not attention_mask.sum(), which was wrong for left-padded batches.

def batched_generate(texts, batch_size=BATCH_SIZE, seed=None):
    if seed is not None:
        set_seed(seed)
    outs = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i : i + batch_size]
        prompts = [build_prompt(t) for t in batch_texts]
        inputs  = tokenizer(
            prompts, return_tensors="pt", padding=True, truncation=True
        ).to(model.device)

        # v2: only pass temperature/top_p when actually sampling
        gen_kwargs = dict(
            max_new_tokens     = MAX_NEW_TOKENS,
            do_sample          = False,
            repetition_penalty = REPETITION_PENALTY,
            pad_token_id       = tokenizer.pad_token_id,
            eos_token_id       = tokenizer.eos_token_id,
        )
        if TEMPERATURE > 0:
            gen_kwargs.update(do_sample=True, temperature=TEMPERATURE, top_p=TOP_P)

        with torch.no_grad():
            gen = model.generate(**inputs, **gen_kwargs)

        # v2: fixed — padded input length is the boundary for all rows in this batch
        padded_input_len = inputs["input_ids"].shape[1]
        for row_idx in range(gen.shape[0]):
            out_ids = gen[row_idx, padded_input_len:]   # was: input_lens[row_idx]
            txt = tokenizer.decode(out_ids, skip_special_tokens=True)
            outs.append(txt)
    return outs

# v5: multi-seed loop (meaningful only when TEMPERATURE > 0; greedy is deterministic)
SEEDS_TO_RUN = SEEDS if TEMPERATURE > 0 else [SEEDS[0]]

all_raw_outputs = {}
for _seed in SEEDS_TO_RUN:
    print(f"\n=== Seed {_seed} (run {SEEDS_TO_RUN.index(_seed)+1}/{len(SEEDS_TO_RUN)}) ===")
    all_raw_outputs[_seed] = batched_generate([r["text"] for r in pool], BATCH_SIZE, seed=_seed)

raw_outputs = all_raw_outputs[SEEDS_TO_RUN[0]]   # primary seed for downstream cells
print("\nGenerated:", len(raw_outputs))
print("\n--- raw_outputs[0] ---")
print(raw_outputs[0][:1000])


In [ ]:
# ====== Parser — broader fence stripping, trailing-comma repair, failure classification ======

def extract_balanced_json_objects(text):
    """Find all top-level balanced {...} objects in text."""
    objs, stack, start = [], 0, None
    in_string, escape = False, False
    for i, ch in enumerate(text):
        if in_string:
            if escape:              escape = False
            elif ch == "\\": escape = True
            elif ch == '"':         in_string = False
            continue
        if   ch == '"': in_string = True
        elif ch == '{': stack += 1; start = start if stack > 1 else i
        elif ch == '}':
            if stack > 0:
                stack -= 1
                if stack == 0 and start is not None:
                    objs.append(text[start : i + 1])
                    start = None
    return objs

def repair_json(s):
    """Remove trailing commas before ] or } — common LLM mistake."""
    return re.sub(r",\s*([}\]])", r"\1", s)

def parse_model_output(raw_text):
    """Returns (parsed_dict_or_None, error_str_or_None)."""
    if not raw_text or not raw_text.strip():
        return None, "empty_output"
    # v2: strip markdown fences anywhere in text
    text = re.sub(r"```(?:json)?", "", raw_text).strip()

    parsed = []
    for frag in extract_balanced_json_objects(text):
        for attempt in (frag, repair_json(frag)):   # try raw, then repaired
            try:
                obj = json.loads(attempt)
                if isinstance(obj, dict):
                    parsed.append(obj)
                    break
            except Exception:
                pass

    if not parsed:
        if "{" in text and "}" not in text:
            return None, "truncated_json"
        if "{" not in text:
            return None, "no_json_found"
        return None, "invalid_json"

    return parsed[-1], None

parsed_preds   = []
failure_counts = Counter()
for txt in raw_outputs:
    obj, err = parse_model_output(txt)
    parsed_preds.append({"pred": obj, "ok": obj is not None, "error": err})
    if err:
        failure_counts[err] += 1

n_ok = sum(p["ok"] for p in parsed_preds)
print(f"Parsed OK: {n_ok} / {len(parsed_preds)}")
if failure_counts:
    print("Failure breakdown:", dict(failure_counts))


In [ ]:
# ====== Normalization helpers — numeric normalization + article stripping ======

WORD_TO_NUM = {
    "zero":"0","one":"1","two":"2","three":"3","four":"4","five":"5",
    "six":"6","seven":"7","eight":"8","nine":"9","ten":"10",
    "eleven":"11","twelve":"12","thirteen":"13","fourteen":"14",
    "fifteen":"15","sixteen":"16","seventeen":"17","eighteen":"18",
    "nineteen":"19","twenty":"20",
    "first":"1","second":"2","third":"3","fourth":"4","fifth":"5",
    "sixth":"6","seventh":"7","eighth":"8","ninth":"9","tenth":"10",
    "eleventh":"11","twelfth":"12","thirteenth":"13","fourteenth":"14",
    "fifteenth":"15","sixteenth":"16","seventeenth":"17","eighteenth":"18",
    "nineteenth":"19","twentieth":"20",
}

def norm_text(v):
    if v is None: return None
    if isinstance(v, str):
        s = v.strip()
        return s if s else None
    if isinstance(v, bool):  return "1" if v else "0"
    if isinstance(v, int):   return str(v)
    if isinstance(v, float): return str(int(v)) if v == int(v) else str(v)
    return None

# v3: slots where comparison should be case-insensitive (categorical labels)
CASE_INSENSITIVE_SLOTS = {"sport_type", "genre", "language", "medium"}


# v5: city alias map — fixes language variants, region-vs-city, and demonym mismatches
CITY_ALIASES = {
    # German/English name pairs
    "wien":               "vienna",
    "viennese":           "vienna",   # demonym used as city
    "austrian capital":   "vienna",
    "muenchen":           "munich",
    "münchen":            "munich",
    "koeln":              "cologne",
    "köln":               "cologne",
    "brüssel":            "brussels",
    "bruxelles":          "brussels",
    # Demonym / descriptor -> proper name
    "scottish capital":   "edinburgh",
    "welsh capital":      "cardiff",
    "polish capital":     "warsaw",
    "danish capital":     "copenhagen",
    "swedish capital":    "stockholm",
    "norwegian capital":  "oslo",
    "dutch capital":      "amsterdam",
    "irish capital":      "dublin",
    "finnish capital":    "helsinki",
    "czech capital":      "prague",
    # Region -> canonical city
    "greater manchester": "manchester",
    "greater london":     "london",
    "greater glasgow":    "glasgow",
    "galway city":        "galway",
}

def _nfc_base(s, slot=None):
    """
    Normalize for comparison only (not stored):
    - Boolean strings: true/false/yes/no/ja/nein -> 1/0
    - Currency stripping: leading €£$¥ and trailing Euro/Pfund/pounds/per...
    - Approximate qualifier stripping for duration: roughly/nearly/about/at least/...
    - Strip leading articles (the/a/an)
    - Normalize hyphens/dashes to spaces (track-and-field -> track and field)
    - Convert number-words to digits
    - Normalize thousands-separated numbers
    - Lowercase for categorical slots (sport_type, genre, language, medium)
    """
    if s is None: return None
    s = s.strip()
    # boolean normalization
    _sl = s.lower()
    if _sl in ("true", "yes", "ja"):   return "1"
    if _sl in ("false", "no", "nein"): return "0"
    # currency stripping: leading symbol then optional digit check
    s = re.sub(r"^[€£$¥]\s*", "", s).strip()
    s = re.sub(r"\s*(euro|euros|pfund|pounds|pound|per\s+\S+.*)$", "", s,
               flags=re.IGNORECASE).strip()
    # approximate qualifier stripping (duration: "roughly four hours" -> "four hours")
    s = re.sub(
        r"^(roughly|nearly|about|approximately|around|"
        r"at\s+least|more\s+than|over|under|up\s+to|"
        r"fast|almost|mindestens|ungef[äa]hr|etwa|rund|bis\s+zu)\s+",
        "", s, flags=re.IGNORECASE).strip()
    # strip leading articles
    s = re.sub(r"^(the|a|an)\s+", "", s, flags=re.IGNORECASE).strip()
    # normalize hyphens/en-dashes/em-dashes to spaces
    s = re.sub(r"[-\u2013\u2014]+", " ", s).strip()
    # word numbers
    if s.lower() in WORD_TO_NUM:
        return WORD_TO_NUM[s.lower()]
    # digit strings with thousands separators (spaces or commas)
    candidate = re.sub(r"[\s,]", "", s)
    if re.fullmatch(r"\d+", candidate):
        return candidate
    # case-insensitive for categorical slots
    if slot in CASE_INSENSITIVE_SLOTS:
        return s.lower()
    # v5: city alias normalisation (case-insensitive)
    if slot == "city":
        _sl = s.lower()
        if _sl in CITY_ALIASES:
            return CITY_ALIASES[_sl]
    return s   # preserve case for proper names

def coerce_list_str(v):
    if v is None: return None
    if isinstance(v, list):
        out = [norm_text(x) for x in v]
        out = [x for x in out if x is not None]
        return out if out else None
    s = norm_text(v)
    return [s] if s is not None else None


# v5.1 surface-form fixes layered on top of _nfc_base:
#   - German thousands separator '.' (95.000 -> 95000)
#   - ordinals / trailing period (76th, 76. -> 76)
#   - currency / unit suffixes (280 SEK -> 280)
#   - city comparison fully case-insensitive (fixes alias-vs-original case)
_ORD_RE = re.compile(r"^(\d+)(st|nd|rd|th|\.)$", re.IGNORECASE)
_CUR_RE = re.compile(
    r"\s*(sek|chf|usd|gbp|eur|kr|kronor|kronen|krone|franken|francs?|"
    r"swedish\s+kronor|swiss\s+francs?|dollars?|euros?|euro|pounds?|pfund)\s*$",
    re.IGNORECASE)

def normalize_for_comparison(s, slot=None):
    base = _nfc_base(s, slot)
    if base is None:
        return None
    out = base
    if re.fullmatch(r"\d{1,3}(\.\d{3})+", out):
        out = out.replace(".", "")
    m = _ORD_RE.match(out)
    if m:
        out = m.group(1)
    out2 = _CUR_RE.sub("", out).strip()
    if out2:
        out = out2
    if slot == "city":
        out = out.lower()
    return out or None


In [ ]:
# ====== Normalize gold + predictions — alias map applied to both ======

def normalize_event_slots(raw_slots, st):
    """Canonicalize slot keys via global + subtype aliases, filter to schema, coerce values."""
    out = {}
    if not isinstance(raw_slots, dict):
        return out
    for raw_k, raw_v in raw_slots.items():
        k = apply_slot_alias_for_subtype(raw_k, st)   # v3: subtype-aware alias
        if k not in subtype_to_slots.get(st, []):
            continue
        kind = slot_kind.get(st, {}).get(k, "scalar")
        val  = coerce_list_str(raw_v) if kind == "list" else norm_text(raw_v)
        if val is not None:
            out[k] = val
    return out

def normalize_gold_record(rec):
    out = []
    for ev in rec.get("events", []):
        bt = canonical_broad_type(ev.get("event_type"))
        st = normalize_subtype(ev.get("event_subtype"))
        if st is None or st not in subtype_to_slots:
            continue
        out.append({"event_type": bt, "event_subtype": st,
                    "slots": normalize_event_slots(ev.get("slots", {}), st)})
    return {"events": out}

def normalize_pred_record(pred):
    if not isinstance(pred, dict):
        return {"events": []}
    events = pred.get("events", [])
    if isinstance(events, dict): events = [events]
    if not isinstance(events, list): return {"events": []}
    out = []
    for ev in events:
        if not isinstance(ev, dict): continue
        bt = canonical_broad_type(ev.get("event_type"))
        st = normalize_subtype(ev.get("event_subtype"))
        if bt is None and st in subtype_to_broad: bt = subtype_to_broad[st]
        if st is None or st not in subtype_to_slots: continue
        if bt not in all_broad_types: bt = subtype_to_broad.get(st)
        out.append({"event_type": bt, "event_subtype": st,
                    "slots": normalize_event_slots(ev.get("slots", {}), st)})
    return {"events": out}

gold_norm_by_id = {r["id"]: normalize_gold_record(gold_by_id[r["id"]]) for r in pool}

norm_preds = []
for rec, pp, raw in zip(pool, parsed_preds, raw_outputs):
    pred = normalize_pred_record(pp["pred"])
    norm_preds.append({"id": rec["id"], "raw_output": raw,
                       "parsed": pp["pred"], "pred": pred, "parse_error": pp["error"]})
pred_norm_by_id = {r["id"]: r["pred"] for r in norm_preds}

print("Normalized:", len(pred_norm_by_id))
print(json.dumps(pred_norm_by_id[pool[0]["id"]], ensure_ascii=False, indent=2))


In [ ]:
# ====== Event matching + metrics ======
# Changes:
#   scalar_equal uses normalize_for_comparison (numeric words, separators, articles)
#   list_f1 likewise normalized
#   Two slot metrics: matched-events-only (v1-compatible) + all-events (v2)
#   Subtype-mismatch diagnostic

def scalar_equal(a, b, slot=None):
    na = normalize_for_comparison(norm_text(a), slot)
    nb = normalize_for_comparison(norm_text(b), slot)
    if na is None and nb is None: return True
    if na is None or nb is None:  return False
    return na == nb

def list_f1(pred_list, gold_list, slot=None):
    pred_set = {normalize_for_comparison(norm_text(x), slot) for x in (pred_list or [])}
    gold_set = {normalize_for_comparison(norm_text(x), slot) for x in (gold_list or [])}
    pred_set.discard(None); gold_set.discard(None)
    tp = len(pred_set & gold_set)
    return tp, len(pred_set - gold_set), len(gold_set - pred_set)

def event_slot_overlap(pred_ev, gold_ev):
    if pred_ev["event_subtype"] != gold_ev["event_subtype"]:
        return -1
    score = 0
    st = gold_ev["event_subtype"]
    for k in subtype_to_slots.get(st, []):
        kind = slot_kind.get(st, {}).get(k, "scalar")
        pv, gv = pred_ev["slots"].get(k), gold_ev["slots"].get(k)
        if kind == "list":
            tp, _, _ = list_f1(pv, gv, slot=k)
            score += tp
        elif pv is not None and gv is not None and scalar_equal(pv, gv, slot=k):
            score += 1
    return score

def greedy_match(pred_events, gold_events):
    candidates = [
        (event_slot_overlap(p, g), i, j)
        for i, p in enumerate(pred_events)
        for j, g in enumerate(gold_events)
        if event_slot_overlap(p, g) >= 0
    ]
    candidates.sort(reverse=True, key=lambda x: (x[0], -x[1], -x[2]))
    pairs, used_p, used_g = [], set(), set()
    for s, i, j in candidates:
        if i in used_p or j in used_g: continue
        pairs.append((i, j, s)); used_p.add(i); used_g.add(j)
    return pairs, used_p, used_g

def safe_div(a, b):
    return a / b if b else 0.0

# ---- accumulate ----
event_tp = event_fp = event_fn = 0
m_tp = m_fp = m_fn = 0          # matched events (v1-compatible)
unmatched_gold_fn = 0            # extra FN from unmatched gold events
slot_rows          = []          # for per-slot breakdown
subtype_mismatch   = Counter()

for rec in pool:
    rid         = rec["id"]
    pred_events = pred_norm_by_id[rid]["events"]
    gold_events = gold_norm_by_id[rid]["events"]
    pairs, used_p, used_g = greedy_match(pred_events, gold_events)

    event_tp += len(pairs)
    event_fp += len(pred_events) - len(used_p)
    event_fn += len(gold_events) - len(used_g)

    # slots on matched pairs
    for i, j, _ in pairs:
        p, g = pred_events[i], gold_events[j]
        st   = g["event_subtype"]
        for slot in subtype_to_slots.get(st, []):
            kind = slot_kind.get(st, {}).get(slot, "scalar")
            pv, gv = p["slots"].get(slot), g["slots"].get(slot)
            if kind == "list":
                tp, fp, fn = list_f1(pv, gv, slot=slot)
            elif pv is None and gv is None:
                tp = fp = fn = 0
            elif scalar_equal(pv, gv, slot=slot):
                tp, fp, fn = 1, 0, 0
            else:
                tp = 0
                fp = 1 if pv is not None else 0
                fn = 1 if gv is not None else 0
            m_tp += tp; m_fp += fp; m_fn += fn
            slot_rows.append({"subtype": st, "slot": slot, "kind": kind,
                               "tp": tp, "fp": fp, "fn": fn})

    # unmatched gold events -> FN for every non-null gold slot
    for j, g in enumerate(gold_events):
        if j in used_g: continue
        st = g["event_subtype"]
        for slot in subtype_to_slots.get(st, []):
            gv = g["slots"].get(slot)
            if gv is None: continue
            fn = len(gv) if isinstance(gv, list) else 1
            unmatched_gold_fn += fn

    # subtype mismatch diagnostic
    for i, p in enumerate(pred_events):
        if i in used_p: continue
        for j, g in enumerate(gold_events):
            if j in used_g: continue
            if p["event_subtype"] != g["event_subtype"]:
                subtype_mismatch[(p["event_subtype"], g["event_subtype"])] += 1

                       "n_pred": len(pred_events), "n_matched": len(pairs),
                       "text": rec["text"][:300]})

# ---- metrics ----
e_prec = safe_div(event_tp, event_tp + event_fp)
e_rec  = safe_div(event_tp, event_tp + event_fn)
e_f1   = safe_div(2 * e_prec * e_rec, e_prec + e_rec)

# v1-compatible: slots on matched events only
s1_prec = safe_div(m_tp, m_tp + m_fp)
s1_rec  = safe_div(m_tp, m_tp + m_fn)
s1_f1   = safe_div(2 * s1_prec * s1_rec, s1_prec + s1_rec)

# v2: includes unmatched gold event FN in recall denominator
s2_prec = s1_prec
s2_rec  = safe_div(m_tp, m_tp + m_fn + unmatched_gold_fn)
s2_f1   = safe_div(2 * s2_prec * s2_rec, s2_prec + s2_rec)

event_metrics = pd.DataFrame([
    {"metric": "events_micro",
     "precision": round(e_prec, 4), "recall": round(e_rec, 4), "f1": round(e_f1, 4)},
    {"metric": "slots_on_matched_events (v1)",
     "precision": round(s1_prec, 4), "recall": round(s1_rec, 4), "f1": round(s1_f1, 4)},
    {"metric": "slots_all_events (v2)",
     "precision": round(s2_prec, 4), "recall": round(s2_rec, 4), "f1": round(s2_f1, 4)},
])

slot_df = (
    pd.DataFrame(slot_rows)
    .groupby(["subtype", "slot", "kind"], as_index=False)[["tp", "fp", "fn"]].sum()
)
slot_df["precision"] = slot_df.apply(lambda r: safe_div(r.tp, r.tp + r.fp), axis=1)
slot_df["recall"]    = slot_df.apply(lambda r: safe_div(r.tp, r.tp + r.fn), axis=1)
slot_df["f1"]        = slot_df.apply(
    lambda r: safe_div(2 * r.precision * r.recall, r.precision + r.recall), axis=1
)
slot_df = slot_df.sort_values(["subtype", "f1", "slot"], ascending=[True, False, True]).reset_index(drop=True)

print(event_metrics.to_string())
print(f"\nTop subtype mismatches (pred -> gold):")
for (ps, gs), cnt in subtype_mismatch.most_common(10):
    print(f"  {ps} -> {gs}: {cnt}")
print("\nPer-slot metrics (top 30):")
display(slot_df.head(30))

# ── v3.3: precision=1.0 disclaimer ──
p1 = (slot_df["precision"] == 1.0) & (slot_df["tp"] > 0)
print(f"\n── Evaluation note ──")
print(f"Slots with precision=1.0: {p1.sum()} / {len(slot_df)}")
print("On this dataset 85%+ of gold values appear exactly once in the article")
print("(verbatim, unique). precision=1.0 reflects dataset difficulty, not model")
print("excellence. See the baseline comparison cell below for LLM lift over no-LLM baselines.")


In [ ]:
# ====== EN vs DE score comparison ======
# Events-micro F1 and Slots-v1 F1 computed SEPARATELY for English and German
# texts, using the same matching + scoring as the main metrics cell above.
lang_by_id = {r["id"]: (r.get("language") or "unknown") for r in pool}

def _score_subset(rids):
    e_tp = e_fp = e_fn = 0
    s_tp = s_fp = s_fn = 0
    for rid in rids:
        pe = pred_norm_by_id[rid]["events"]
        ge = gold_norm_by_id[rid]["events"]
        pairs, used_p, used_g = greedy_match(pe, ge)
        e_tp += len(pairs)
        e_fp += len(pe) - len(used_p)
        e_fn += len(ge) - len(used_g)
        for i, j, _ in pairs:
            p, g = pe[i], ge[j]
            st = g["event_subtype"]
            for slot in subtype_to_slots.get(st, []):
                kind = slot_kind.get(st, {}).get(slot, "scalar")
                pv, gv = p["slots"].get(slot), g["slots"].get(slot)
                if kind == "list":
                    tp, fp, fn = list_f1(pv, gv, slot=slot)
                elif pv is None and gv is None:
                    tp = fp = fn = 0
                elif scalar_equal(pv, gv, slot=slot):
                    tp, fp, fn = 1, 0, 0
                else:
                    tp = 0
                    fp = 1 if pv is not None else 0
                    fn = 1 if gv is not None else 0
                s_tp += tp; s_fp += fp; s_fn += fn
    return e_tp, e_fp, e_fn, s_tp, s_fp, s_fn

def _prf(tp, fp, fn):
    p = safe_div(tp, tp + fp)
    r = safe_div(tp, tp + fn)
    return p, r, safe_div(2 * p * r, p + r)

_lang_rows = []
for _lang in sorted({lang_by_id[r["id"]] for r in pool}) + ["all"]:
    _rids = [r["id"] for r in pool if _lang == "all" or lang_by_id[r["id"]] == _lang]
    e_tp, e_fp, e_fn, s_tp, s_fp, s_fn = _score_subset(_rids)
    ep, er, ef1 = _prf(e_tp, e_fp, e_fn)
    sp, sr, sf1 = _prf(s_tp, s_fp, s_fn)
    _lang_rows.append({
        "language": _lang, "n_texts": len(_rids),
        "event_P": round(ep, 4), "event_R": round(er, 4), "event_F1": round(ef1, 4),
        "slot_P": round(sp, 4), "slot_R": round(sr, 4), "slot_F1": round(sf1, 4),
    })

lang_compare_df = pd.DataFrame(_lang_rows)
print("=== EN vs DE comparison (events-micro + slots-v1, matched events) ===")
display(lang_compare_df)

In [ ]:
# ====== Macro slot-F1 excluding F1 == 0.0 slots (all languages) ======
# Many (subtype, slot) rows score F1 = 0.0; most are simply absent from the
# dataset (tp = fp = fn = 0) and reflect schema coverage, not model failure.
# This reports the macro-averaged per-slot F1 with the zero-F1 rows removed.
_total      = len(slot_df)
_present    = slot_df[(slot_df.tp + slot_df.fp + slot_df.fn) > 0]
_absent     = _total - len(_present)
_nonzero    = slot_df[slot_df.f1 > 0.0]

print("=== Macro slot-F1 (mean over (subtype, slot) rows) ===")
print(f"Slots total:                        {_total}")
print(f"  - absent (tp=fp=fn=0):            {_absent}")
print(f"  - present but scored F1 = 0.0:    {len(_present) - len(_nonzero)}")
print(f"  - scored F1 > 0.0:                {len(_nonzero)}")
print()
print(f"Macro slot-F1 (all rows):           {slot_df.f1.mean():.4f}")
print(f"Macro slot-F1 (present rows):       {_present.f1.mean():.4f}")
print(f"Macro slot-F1 (excluding F1=0.0):   {_nonzero.f1.mean():.4f}")

In [ ]:
# ====== Multi-seed metric averaging ======
# If TEMPERATURE > 0, re-runs full parse + eval for each seed and reports mean +/- std.
# If TEMPERATURE == 0, greedy decoding is deterministic -- single run is sufficient.

import numpy as _np

if len(SEEDS_TO_RUN) > 1:
    seed_metrics = []
    for _seed in SEEDS_TO_RUN:
        _raws = all_raw_outputs[_seed]
        _parsed = [parse_output(r) for r in _raws]
        _norm_preds_s = []
        for rec, pp, raw in zip(pool, _parsed, _raws):
            pred = normalize_pred_record(pp["pred"])
            _norm_preds_s.append({"id": rec["id"], "pred": pred})
        _pred_norm_s = {r["id"]: r["pred"] for r in _norm_preds_s}

        _tp = _fp = _fn = _m_tp = _m_fp = _m_fn = _ufn = 0
        for rec in pool:
            rid = rec["id"]
            pe = _pred_norm_s[rid]["events"]
            ge = gold_norm_by_id[rid]["events"]
            pairs, up, ug = greedy_match(pe, ge)
            _tp += len(pairs); _fp += len(pe) - len(up); _fn += len(ge) - len(ug)
            for i, j, _ in pairs:
                p, g = pe[i], ge[j]
                st = g["event_subtype"]
                for slot in subtype_to_slots.get(st, []):
                    kind = slot_kind.get(st, {}).get(slot, "scalar")
                    pv, gv = p["slots"].get(slot), g["slots"].get(slot)
                    if kind == "list":
                        tp, fp, fn = list_f1(pv, gv, slot=slot)
                    elif pv is None and gv is None:
                        tp = fp = fn = 0
                    elif scalar_equal(pv, gv, slot=slot):
                        tp, fp, fn = 1, 0, 0
                    else:
                        tp = 0
                        fp = 1 if pv is not None else 0
                        fn = 1 if gv is not None else 0
                    _m_tp += tp; _m_fp += fp; _m_fn += fn
            for j, g in enumerate(ge):
                if j in ug: continue
                st = g["event_subtype"]
                for slot in subtype_to_slots.get(st, []):
                    gv = g["slots"].get(slot)
                    if gv is None: continue
                    _ufn += len(gv) if isinstance(gv, list) else 1

        ep = safe_div(_tp, _tp + _fp); er = safe_div(_tp, _tp + _fn)
        ef1 = safe_div(2 * ep * er, ep + er)
        s1p = safe_div(_m_tp, _m_tp + _m_fp); s1r = safe_div(_m_tp, _m_tp + _m_fn)
        s1f1 = safe_div(2 * s1p * s1r, s1p + s1r)
        s2r = safe_div(_m_tp, _m_tp + _m_fn + _ufn)
        s2f1 = safe_div(2 * s1p * s2r, s1p + s2r)
        seed_metrics.append({"seed": _seed,
                             "event_f1": ef1, "slot_v1_f1": s1f1, "slot_v2_f1": s2f1,
                             "event_prec": round(ep, 4), "event_rec": round(er, 4)})

    seed_df = pd.DataFrame(seed_metrics)
    print("=== Per-seed metrics ===")
    print(seed_df.to_string(index=False))
    print(f"\nEvent F1:   {_np.mean(seed_df.event_f1):.4f} +/- {_np.std(seed_df.event_f1):.4f}")
    print(f"Slot v1 F1: {_np.mean(seed_df.slot_v1_f1):.4f} +/- {_np.std(seed_df.slot_v1_f1):.4f}")
    print(f"Slot v2 F1: {_np.mean(seed_df.slot_v2_f1):.4f} +/- {_np.std(seed_df.slot_v2_f1):.4f}")

    avg_metrics = pd.DataFrame([
        {"metric": "events_micro",
         "precision": round(_np.mean(seed_df.event_prec), 4),
         "recall":    round(_np.mean(seed_df.event_rec), 4),
         "f1":        round(_np.mean(seed_df.event_f1), 4),
         "f1_std":    round(_np.std(seed_df.event_f1), 4)},
        {"metric": "slots_v1",
         "precision": None, "recall": None,
         "f1":        round(_np.mean(seed_df.slot_v1_f1), 4),
         "f1_std":    round(_np.std(seed_df.slot_v1_f1), 4)},
        {"metric": "slots_v2",
         "precision": None, "recall": None,
         "f1":        round(_np.mean(seed_df.slot_v2_f1), 4),
         "f1_std":    round(_np.std(seed_df.slot_v2_f1), 4)},
    ])
    avg_path = MET_DIR / f"multiseed_avg_{MODEL_TAG}_{ts}.csv"
    avg_metrics.to_csv(avg_path, index=False)
    print(f"Saved averaged metrics: {avg_path}")
else:
    print(f"Multi-seed: skipped -- TEMPERATURE=0.0, greedy decoding is deterministic.")
    print(f"Primary seed: {SEEDS_TO_RUN[0]}. Set TEMPERATURE > 0 and re-run to enable averaging.")


In [ ]:
# ====== Baseline comparison (no-LLM) ======
# ── inline baselines (keyword + regex, no I/O dependencies) ──
    import re as _re
    from collections import defaultdict as _dd

    _SUBTYPE_PATTERNS = [
        # mobility
        ("TransitDisruption",      r"\b(disruption|suspension|signal.fault|strike|derailment|sperrung|betriebsstörung|streik|ausfall)\b"),
        ("TransitInfrastructure",  r"\b(line extension|groundbreaking|metro extension|tram extension|new station|streckenverlängerung|neubau)\b"),
        ("TransitServiceChange",   r"\b(fare (change|increase)|extra (service|train)|shuttle|diversion|frequency increase|tarif)\b"),
        # sports
        ("FootballMatch",          r"\b(football|soccer|fussball|bundesliga|premier league|la liga|cup (match|tie|final)|derby|league match)\b"),
        ("TeamMatch",              r"\b(rugby|handball|ice hockey|basketball|volleyball|water polo|fencing)\b"),
        ("RaceEvent",              r"\b(marathon|road race|cycling stage|tennis (match|final)|rowing|regatta|athletics|sprint)\b"),
        # cultural
        ("Premiere",               r"\b(premiere|uraufführung|world premiere|erstaufführung)\b"),
        ("Exhibition",             r"\b(exhibition|retrospective|ausstellung|photography show|sculpture show)\b"),
        ("Festival",               r"\b(festival|book fair|buchmesse|kunstmesse|art fair|messe)\b"),
        ("Concert",                r"\b(concert|recital|orchestral|konzert|philharmonic|symphony)\b"),
    ]

    QUOTE_RE = _re.compile(r'[""„]([^""„]{2,120}?)[""„]')

    def _detect(text):
        t = text.lower()
        for st, pat in _SUBTYPE_PATTERNS:
            if _re.search(pat, t): return st
        return None

    def _fill(st, text):
        slots = {}
        if "event_name" in subtype_to_slots.get(st, []):
            m = QUOTE_RE.search(text); slots["event_name"] = m.group(1) if m else None
        if "city" in subtype_to_slots.get(st, []):
            m = _re.search(r"\bin\s+([A-ZÄÖÜ][\w\-]+(?:\s+[A-ZÄÖÜ][\w\-]+){0,2})", text)
            if m: slots["city"] = m.group(1)
        for ds in ("start_date", "event_date"):
            if ds in subtype_to_slots.get(st, []):
                m = _re.search(r"\b(\d{1,2}[\s.]?(?:January|February|March|April|May|June|"
                               r"July|August|September|October|November|December|"
                               r"Januar|Februar|März|Mai|Juni|Juli|August|September|"
                               r"Oktober|November|Dezember)\s*\d{4}|"
                               r"(?:Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday|"
                               r"Montag|Dienstag|Mittwoch|Donnerstag|Freitag|Samstag|Sonntag))\b", text)
                if m: slots[ds] = m.group(1)
                break
        if "final_score" in subtype_to_slots.get(st, []):
            m = _re.search(r"\b(\d{1,2}[-–:]\d{1,2})\b", text)
            if m: slots["final_score"] = m.group(1)
        return {k: v for k, v in slots.items() if v}

    # baselines
    _b1 = {r["id"]: [] for r in pool}
    _b2 = {}
    for r in pool:
        st = _detect(r["text"])
        if st:
            _b2[r["id"]] = [{"event_subtype": st,
                              "event_type": subtype_to_broad.get(st,"cultural"),
                              "slots": {}}]
        else:
            _b2[r["id"]] = []
    _b3 = {}
    for r in pool:
        st = _detect(r["text"])
        if not st: _b3[r["id"]] = []; continue
        _b3[r["id"]] = [{"event_subtype": st,
                          "event_type": subtype_to_broad.get(st,"cultural"),
                          "slots": _fill(st, r["text"])}]

    def _eval(preds_by_id):
        etp=efp=efn=stp=sfp=sfn=ugfn=0
        for r in pool:
            rid=r["id"]
            pe=preds_by_id.get(rid,[])
            ge=gold_norm_by_id[rid]["events"]
            pairs,up,ug=greedy_match(pe,ge)
            etp+=len(pairs); efp+=len(pe)-len(up); efn+=len(ge)-len(ug)
            for i,j,_ in pairs:
                p,g=pe[i],ge[j]; st=g["event_subtype"]
                for sl in subtype_to_slots.get(st,[]):
                    kd=slot_kind.get(st,{}).get(sl,"scalar")
                    pv,gv=p["slots"].get(sl),g["slots"].get(sl)
                    if kd=="list": tp2,fp2,fn2=list_f1(pv,gv,slot=sl)
                    elif pv is None and gv is None: tp2=fp2=fn2=0
                    elif scalar_equal(pv,gv,slot=sl): tp2,fp2,fn2=1,0,0
                    else: tp2=0; fp2=1 if pv else 0; fn2=1 if gv else 0
                    stp+=tp2; sfp+=fp2; sfn+=fn2
            for j,g in enumerate(ge):
                if j in ug:
                    st=g["event_subtype"]
                    for sl in subtype_to_slots.get(st,[]):
                        gv=g["slots"].get(sl)
                        if gv: ugfn+=len(gv) if isinstance(gv,list) else 1
        ep=safe_div(etp,etp+efp); er=safe_div(etp,etp+efn)
        sp=safe_div(stp,stp+sfp); sr=safe_div(stp,stp+sfn)
        sr2=safe_div(stp,stp+sfn+ugfn)
        return {"ev_f1":safe_div(2*ep*er,ep+er),
                "sl_f1":safe_div(2*sp*sr,sp+sr),
                "sl_f1_all":safe_div(2*sp*sr2,sp+sr2)}

    print("\n── No-LLM baselines (same eval logic) ──")
    print(f"{'Baseline':45s}  ev_F1   slt_F1(matched)  slt_F1(all)")
    print("-"*78)
    for _name, _preds in [
        ("(1) always no events", _b1),
        ("(2) keyword subtype, no slots", _b2),
        ("(3) (2) + first quoted string + regex slots", _b3),
    ]:
        _m = _eval(_preds)
        print(f"{_name:45s}  {_m['ev_f1']:.4f}   {_m['sl_f1']:.4f}            {_m['sl_f1_all']:.4f}")

    _llm_ev = e_f1; _llm_sl = s1_f1
    _lift_ev  = _llm_ev - _eval(_b3)["ev_f1"]
    _lift_sl  = _llm_sl - _eval(_b3)["sl_f1"]
    print(f"\nLLM (this run)                                  "
          f"  {_llm_ev:.4f}   {_llm_sl:.4f}")
    print(f"\nLLM lift over baseline (3): ev_F1 = {_lift_ev:+.4f}   slt_F1 = {_lift_sl:+.4f}")
    print("\nNOTE: scores above baseline reflect genuine LLM contribution.")
    print("Raw LLM scores alone are inflated (verbatim values, 85%+ unique per article).")


In [ ]:
# ====== Save outputs ======
ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

pred_path          = PRED_DIR / f"predictions_{MODEL_TAG}_{ts}.jsonl"
event_metrics_path = MET_DIR / f"event_metrics_{MODEL_TAG}_{ts}.csv"
slot_path          = MET_DIR / f"slot_metrics_{MODEL_TAG}_{ts}.csv"

with open(pred_path, "w", encoding="utf-8") as f:
    for row in norm_preds:
        f.write(json.dumps(row, ensure_ascii=False) + "
")

event_metrics.to_csv(event_metrics_path, index=False, encoding="utf-8")
slot_df.to_csv(slot_path, index=False, encoding="utf-8")

print("Saved:")
for p in [pred_path, event_metrics_path, slot_path]:
    print(" ", p)
